# Tiếp tục Experiment 7 trên Kaggle

Bật **Internet** và dùng CPU. Thêm `exp7_checkpoint.zip` vào Kaggle như một dataset riêng tư rồi gắn dataset vào notebook. GitHub cung cấp mã nguồn, còn checkpoint chứa CSV episode và các cấu hình tuning của lượt chạy hiện tại.

Notebook chạy Robust-UCT với TV epsilon 0.1 trên lưới exec 0.1–0.8. Sáu planner còn lại giữ cấu hình tuning cũ và chỉ chạy các mức còn thiếu đến 0.8. Mọi lượt chạy dùng N=2000. Kết quả được cập nhật vào các đường dẫn CSV hiện có. Tên thư mục cũ của Robust-UCT có thể vẫn là `eps0.2`; cột `uncertainty_budget` ghi epsilon thực tế.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Thangcoding/Graph-Robust-Planning.git"
GIT_REF = "c0c84ae46f0c11b2a51a8dcc333a2c39d3ffa6fb"
PRIVATE_REPO = False  # Nếu cần, thêm Kaggle Secret GITHUB_TOKEN có quyền đọc repo.
REPO = Path("/tmp") / f"graph-robust-planning-exp7-{GIT_REF[:12]}"
WORK = Path("/kaggle/working")
RESULTS = WORK / "experiement_7"
CHECKPOINT_NAME = "exp7_checkpoint.zip"
WORKERS = len(os.sched_getaffinity(0))

# Có thể giới hạn phạm vi cho một phiên Kaggle.
PLANS = {"gambler": (0.4, 0.6, 0.8), "frozen_lake": (0.3, 0.4, 0.5),
         "cliffwalking": (0.3, 0.4, 0.5)}
ENVS = list(PLANS)
ALGORITHMS = {
    "robust-uct": ("robust_uct", 0.1),
    "uct": ("uct", None),
    "stochastic-power-uct": ("st_power_uct", None),
    "gs-power-uct": ("gs_power_uct", None),
    "gsf-power-uct": ("gs_power_uct_f", None),
    "gs-robust-uct": ("gs_robust_uct", 0.2),
    "gsf-robust-uct": ("gs_robust_uct_f", 0.2),
}
RUN_ALGOS = list(ALGORITHMS)

In [ ]:
git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
if PRIVATE_REPO:
    from base64 import b64encode
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    auth = b64encode(f"x-access-token:{token}".encode()).decode()
    git_env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"Authorization: Basic {auth}")
    del token, auth

REPO.mkdir(parents=True, exist_ok=True)
def git(*args, **kwargs):
    return subprocess.run(["git", "-C", str(REPO), *args], check=True, env=git_env, text=True, **kwargs)

try:
    git("init", "--quiet")
    remotes = git("remote", capture_output=True).stdout.split()
    git("remote", "set-url" if "origin" in remotes else "add", "origin", REPO_URL)
    git("config", "remote.origin.promisor", "true")
    git("config", "remote.origin.partialclonefilter", "blob:none")
    git("sparse-checkout", "set", "--no-cone", "/src/", "/requirements.txt",
        "/experiments/*.py")
    git("fetch", "--depth=1", "--filter=blob:none", "origin", GIT_REF)
    git("checkout", "--detach", "FETCH_HEAD")
    revision = git("rev-parse", "HEAD", capture_output=True).stdout.strip()
except subprocess.CalledProcessError:
    raise RuntimeError("GitHub fetch/checkout thất bại. Kiểm tra Internet, commit và quyền đọc repo.") from None
finally:
    del git_env

subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "requirements.txt")], check=True)

## Khôi phục checkpoint

Để tiếp tục qua phiên Kaggle mới, lưu hoặc tải `exp7_checkpoint.zip` từ output rồi gắn checkpoint mới nhất. Notebook bỏ qua file đã có trong thư mục làm việc để giữ tiến độ nếu bạn chạy lại các cell trong cùng phiên.

In [ ]:
from zipfile import ZipFile

RESULTS.mkdir(parents=True, exist_ok=True)
checkpoints = sorted(Path("/kaggle/input").rglob(CHECKPOINT_NAME))
if len(checkpoints) > 1:
    raise ValueError("Chỉ gắn một dataset checkpoint để chọn đúng phiên bản.")
if checkpoints:
    with ZipFile(checkpoints[0]) as archive:
        for entry in archive.infolist():
            path = Path(entry.filename)
            if entry.is_dir() or path.is_absolute() or ".." in path.parts:
                continue
            target = RESULTS / path
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(entry) as source, target.open("wb") as dest:
                    shutil.copyfileobj(source, dest)
if not all((RESULTS / env).is_dir() for env in ENVS):
    raise FileNotFoundError(f"Hãy gắn dataset chứa {CHECKPOINT_NAME} rồi chạy lại ô này.")

## Chạy phần còn thiếu và cập nhật CSV

Mỗi cấu hình hoàn tất được ghép ngay vào `summary.csv` và `episodes.csv` trong cây thư mục `run`. Notebook tạo lại ZIP checkpoint sau từng cấu hình, ở `/kaggle/working/exp7_checkpoint.zip`.

In [ ]:
import pandas as pd

FULL_GRID = set(range(1, 9))
NAMES = ("episodes.csv", "summary.csv")
runtime_env = {**os.environ, "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1",
               "MKL_NUM_THREADS": "1"}

def grid(rows):
    return set((rows.theta_exec * 10).round().astype(int))

def write_csv(rows, path):
    temp = path.with_suffix(".tmp")
    rows.to_csv(temp, index=False)
    temp.replace(path)

def checkpoint():
    from zipfile import ZIP_DEFLATED
    temporary = WORK / "exp7_checkpoint.tmp"
    with ZipFile(temporary, "w", compression=ZIP_DEFLATED, compresslevel=1) as archive:
        for path in sorted(RESULTS.rglob("*")):
            if path.name in (*NAMES, "top3.json"):
                archive.write(path, path.relative_to(RESULTS))
    temporary.replace(WORK / CHECKPOINT_NAME)

def destination(base, folder):
    matches = list((base / "run" / folder).rglob("summary.csv"))
    if len(matches) != 1 or not matches[0].with_name("episodes.csv").exists():
        raise ValueError(f"Checkpoint thiếu hoặc trùng CSV trong {base / 'run' / folder}")
    return matches[0].parent

def valid_rows(rows, name, env, algo, plan, points, eps):
    keys = ["theta_exec", "seed", "episode"] if name == "episodes.csv" else ["theta_exec"]
    return (grid(rows) == points and len(rows) == len(points) * (200 if len(keys) > 1 else 1)
            and not rows.duplicated(keys).any() and rows.theta_plan.eq(plan).all()
            and rows.config_id.eq(algo).all() and rows.num_simulations.eq(2000).all()
            and (eps is None or rows.uncertainty_budget.eq(eps).all()))

def run_setting(env, plan, algo):
    folder, eps = ALGORITHMS[algo]
    base = RESULTS / env / f"plan{plan:g}"
    dest = destination(base, folder)
    old = {name: pd.read_csv(dest / name) for name in NAMES}
    if all(valid_rows(rows, name, env, algo, plan, FULL_GRID, eps) for name, rows in old.items()):
        print(f"Đã đủ {env} plan={plan:g} {algo}", flush=True)
        return

    replace = algo == "robust-uct"
    missing = FULL_GRID if replace else FULL_GRID - (grid(old[NAMES[0]]) & grid(old[NAMES[1]]))
    work_run = base / "run_exec08"
    relative = Path(folder) / env / "sims2000"
    eps_args = []
    if eps is not None:
        relative /= f"eps{eps:g}"
        eps_args = ["--eps", f"{eps:g}"]
    tune_dir = base / "tune"
    args = ["--env", env, "--algo", algo, "--sims", "2000", "--workers", str(WORKERS),
            "--theta-plan", str(plan), "--tune-root", str(tune_dir), *eps_args]
    theta_args = ["--theta-exec", *(f"{i / 10:g}" for i in sorted(missing))]
    if not (tune_dir / relative / "top3.json").is_file():
        if not replace:
            raise FileNotFoundError(tune_dir / relative / "top3.json")
        subprocess.run([sys.executable, "-u", "experiments/tune.py", *args, *theta_args],
                       cwd=REPO, env=runtime_env, check=True)
        checkpoint()

    staged = work_run / relative
    reused = {}
    if all((staged / name).is_file() for name in NAMES):
        for name in NAMES:
            rows = pd.read_csv(staged / name)
            reused[name] = rows.loc[(rows.theta_exec * 10).round().astype(int).isin(missing)]
    if not reused or not all(valid_rows(rows, name, env, algo, plan, missing, eps)
                              for name, rows in reused.items()):
        subprocess.run([sys.executable, "-u", "experiments/run.py", *args, *theta_args,
                        "--run-root", str(work_run)], cwd=REPO, env=runtime_env, check=True)
        reused = {name: pd.read_csv(staged / name) for name in NAMES}

    updated = {}
    for name, rows in reused.items():
        if not valid_rows(rows, name, env, algo, plan, missing, eps):
            raise ValueError(f"Kết quả mới không khớp cấu hình {staged / name}")
        if set(old[name].columns) != set(rows.columns):
            raise ValueError(f"Cột CSV không khớp giữa {dest} và {staged}")
        retained = old[name].loc[~(old[name].theta_exec * 10).round().astype(int).isin(missing)]
        combined = rows if replace else pd.concat([retained, rows], ignore_index=True)
        if not valid_rows(combined, name, env, algo, plan, FULL_GRID, eps):
            raise ValueError(f"Kết quả ghép chưa đủ hoặc bị trùng ở {dest / name}")
        sort_by = ["theta_exec", "seed", "episode"] if name == "episodes.csv" else ["theta_exec"]
        updated[name] = combined[old[name].columns].sort_values(sort_by)
    for name, rows in updated.items():
        write_csv(rows, dest / name)
    checkpoint()
    print(f"Đã cập nhật {dest}", flush=True)

In [ ]:
for env in ENVS:
    for plan in PLANS[env]:
        for algo in RUN_ALGOS:
            run_setting(env, plan, algo)
checkpoint()
print(f"Kết quả: {RESULTS}")
print(f"Checkpoint: {WORK / CHECKPOINT_NAME}")